In [33]:
import pandas as pd

In [34]:
train = pd.read_csv('../data/raw/train.csv')
train.set_index('id', inplace=True)

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
for col in train.columns:
    print(f'{col}: Type is {train[col].dtype} and has {train[col].nunique()} unique values')
    print()

Soil_Type: Type is object and has 4 unique values

Soil_pH: Type is float64 and has 341 unique values

Soil_Moisture: Type is float64 and has 5223 unique values

Organic_Carbon: Type is float64 and has 131 unique values

Electrical_Conductivity: Type is float64 and has 341 unique values

Temperature_C: Type is float64 and has 2934 unique values

Humidity: Type is float64 and has 6475 unique values

Rainfall_mm: Type is float64 and has 19308 unique values

Sunlight_Hours: Type is float64 and has 701 unique values

Wind_Speed_kmh: Type is float64 and has 1935 unique values

Crop_Type: Type is object and has 6 unique values

Crop_Growth_Stage: Type is object and has 4 unique values

Season: Type is object and has 3 unique values

Irrigation_Type: Type is object and has 4 unique values

Water_Source: Type is object and has 4 unique values

Field_Area_hectare: Type is float64 and has 1466 unique values

Mulching_Used: Type is object and has 2 unique values

Previous_Irrigation_mm: Type is f

In [37]:
def preprocess_data(df):
    df = df.copy()
    
    # Handle missing values
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

    categorical_cols = df.select_dtypes(include=['object']).columns
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    return df

In [38]:
y = train['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})

train_data = train.drop(columns=['Irrigation_Need'])

X_train, X_val, y_train, y_val = train_test_split(train_data, y, test_size=0.2, random_state=42, stratify=y)

X_train_processed = preprocess_data(X_train)
X_val_processed = preprocess_data(X_val)

In [39]:
X_train_processed

,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,...,Irrigation_Type_Rainfed,Irrigation_Type_Sprinkler,Water_Source_Rainwater,Water_Source_Reservoir,Water_Source_River,Mulching_Used_Yes,Region_East,Region_North,Region_South,Region_West
id,,,,,,,,,,,,,,,,,,,,,
197153,5.14,47.02,0.73,3.34,39.81,40.45,1589.81,6.97,19.65,8.23,...,False,False,False,True,False,False,False,False,True,False
268620,6.70,12.50,0.92,0.98,23.54,83.52,2056.19,4.43,16.17,10.81,...,True,False,True,False,False,False,True,False,False,False
475375,7.72,40.13,0.79,1.28,40.34,39.33,1584.61,9.81,1.97,7.03,...,False,True,False,False,False,True,False,False,False,False
275194,5.31,52.10,1.25,0.24,35.16,56.51,990.79,5.17,18.71,6.76,...,True,False,False,False,True,True,False,False,True,False
71652,6.85,54.62,1.51,0.69,36.11,84.59,1808.92,6.88,9.98,7.90,...,False,True,False,True,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247797,5.89,29.59,0.44,3.07,31.85,86.74,2074.33,7.51,17.96,2.57,...,True,False,False,False,True,True,True,False,False,False
287540,5.44,37.96,1.56,1.10,26.38,38.73,2172.66,7.57,8.73,4.09,...,True,False,False,False,False,True,True,False,False,False
169327,5.43,20.27,1.18,1.25,34.79,47.17,1043.44,4.44,13.50,1.61,...,False,False,True,False,False,True,False,False,True,False


In [41]:
y_train

id
197153    1
268620    1
475375    0
275194    0
71652     0
         ..
247797    0
287540    0
169327    1
169581    0
129019    1
Name: Irrigation_Need, Length: 504000, dtype: int64

In [43]:
import xgboost as xgb

model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_estimators=10000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    colsample_bylevel=0.8,
    min_child_weight=1,
    min_split_loss=0,
    reg_lambda=1,
    reg_alpha=0
    )

In [44]:
model.fit(X_train_processed, y_train, eval_set=[(X_val_processed, y_val)], verbose=100)

[0]	validation_0-mlogloss:0.78335


d:\Kaggle\Predicting_Irrigation_Need_S6E4\.venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [23:37:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[100]	validation_0-mlogloss:0.35054
[200]	validation_0-mlogloss:0.19883
[300]	validation_0-mlogloss:0.12978
[400]	validation_0-mlogloss:0.09732
[500]	validation_0-mlogloss:0.08088
[600]	validation_0-mlogloss:0.07232
[700]	validation_0-mlogloss:0.06776
[800]	validation_0-mlogloss:0.06529
[900]	validation_0-mlogloss:0.06367
[1000]	validation_0-mlogloss:0.06253
[1100]	validation_0-mlogloss:0.06169
[1200]	validation_0-mlogloss:0.06105
[1300]	validation_0-mlogloss:0.06054
[1400]	validation_0-mlogloss:0.06011
[1500]	validation_0-mlogloss:0.05977
[1600]	validation_0-mlogloss:0.05945
[1700]	validation_0-mlogloss:0.05917
[1800]	validation_0-mlogloss:0.05893
[1900]	validation_0-mlogloss:0.05870
[2000]	validation_0-mlogloss:0.05851
[2100]	validation_0-mlogloss:0.05833
[2200]	validation_0-mlogloss:0.05818
[2300]	validation_0-mlogloss:0.05802
[2400]	validation_0-mlogloss:0.05788
[2500]	validation_0-mlogloss:0.05777
[2600]	validation_0-mlogloss:0.05766
[2700]	validation_0-mlogloss:0.05755
[2800]	val

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,0.8
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from 

In [46]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score

y_val_pred = model.predict(X_val_processed)
print("Classification Report:")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred))
print("F1 Score (macro):", f1_score(y_val, y_val_pred, average='macro'))
y_val_proba = model.predict_proba(X_val_processed)
print("ROC AUC Score:", roc_auc_score(y_val, y_val_proba, multi_class='ovr', average='macro'))

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     73983
           1       0.98      0.98      0.98     47815
           2       0.97      0.91      0.94      4202

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000

Confusion Matrix:
[[73600   383     0]
 [ 1043 46644   128]
 [    0   381  3821]]
Accuracy: 0.9846428571428572
Balanced Accuracy: 0.9598872677717897
F1 Score (macro): 0.969212800263829
ROC AUC Score: 0.9976814228709427


In [47]:
test = pd.read_csv('../data/raw/test.csv')
test.set_index('id', inplace=True)

In [48]:
test_preprocessed = preprocess_data(test)

In [49]:
pred = model.predict(test_preprocessed)

In [52]:
pred = pd.Series(pred, index=test.index, name='Irrigation_Need')
pred = pred.map({0: 'Low', 1: 'Medium', 2: 'High'})
pred.to_csv('../data/processed/submission.csv', header=True)